# Phase 2 Notebook: Candidate Generation Fernsehserien.de
This notebook orchestrates the full fernsehserien.de candidate-generation pipeline.
It supports discovery, extraction, normalization, and replay across all eligible programs in setup input, with configurable runtime budgets.

## How This Implementation Works (Quick Guide)

This notebook runs one deterministic fernsehserien workflow per execution:
1. Resolve known episode leaf URLs.
2. Fetch missing leaf pages (cache-first), then continue resolution.
3. Fetch additional episode URLs from index traversal as needed, then continue leaf fetching.
4. Normalize discovered data.
5. Build and verify projections.

Network behavior is controlled only by `MAX_NETWORK_CALLS`:
- `0`: cache-only execution (no new network calls).
- `>0`: bounded network execution.
- `<0`: unlimited network budget.

### Where Sequence Numbers Are Persisted
The canonical sequence numbers are stored in the append-only event log:
- `data/20_candidate_generation/fernsehserien_de/chunks/chunk_000001.jsonl`
- each event contains `sequence_num`.

### Per-Handler Progress Table
The projection handler persists progress to:
- `data/20_candidate_generation/fernsehserien_de/eventhandler.csv`

Schema:
- `handler_name,last_processed_sequence,artifact_path,updated_at`

Current handler behavior:
- reads `last_processed_sequence` for each projection handler,
- processes only events with higher sequence numbers,
- writes updated projections, then updates `eventhandler.csv`.

This gives incremental resume behavior while keeping the event log as source of truth.

## 1) Project Setup
Resolve repository root and make `speakermining/src` importable for full-pipeline orchestration.

In [ ]:
# Optional projection reset only: cache pages and events must never be deleted.
import shutil
from pathlib import Path


RESET_PROJECTIONS = False


def find_repo_root_for_cleanup(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "data").exists() and (cur / "speakermining" / "src").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError("Repository root not found for cleanup.")


cleanup_root = find_repo_root_for_cleanup(Path.cwd())
runtime_root = cleanup_root / "data" / "20_candidate_generation" / "fernsehserien_de"
projection_root = runtime_root / "projections"

if RESET_PROJECTIONS:
    if projection_root.exists():
        shutil.rmtree(projection_root)
        print(f"Deleted derived projections only: {projection_root}")
    else:
        print("No projections directory to delete")
else:
    print("Projection reset skipped (RESET_PROJECTIONS=False); cache/events are preserved")

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "data").exists() and (cur / "speakermining" / "src").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError("Repository root not found.")


ROOT = find_repo_root(Path.cwd())
SRC = ROOT / "speakermining" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# ROOT

## 2) Configure Runtime Parameters
Set only the core runtime parameters for production execution.
- `MAX_NETWORK_CALLS` controls cache-only vs bounded/unlimited network behavior.
- `QUERY_DELAY_SECONDS` controls request pacing.
- `USER_AGENT` identifies this pipeline to the service.

In [ ]:
from process.notebook_event_log import get_or_create_notebook_logger
from process.candidate_generation.fernsehserien_de import FernsehserienRunConfig
import pandas as pd

NOTEBOOK_ID = "notebook_22_candidate_generation_fernsehserien_de"
logger = get_or_create_notebook_logger(ROOT, NOTEBOOK_ID)

programs_path = ROOT / "data" / "00_setup" / "broadcasting_programs.csv"
programs_df = pd.read_csv(programs_path)
fernsehserien_ids = programs_df["fernsehserien_de_id"].fillna("").astype(str).str.strip()
eligible_programs_df = programs_df[
    fernsehserien_ids.ne("")
    & (~fernsehserien_ids.str.upper().isin({"NONE", "NULL", "NAN"}))
] .copy()

# Core runtime configuration
MAX_NETWORK_CALLS = 10  # 0=cache-only, >0=bounded network, <0=unlimited
QUERY_DELAY_SECONDS = 1.0
USER_AGENT = "speaker-mining/0.1 (fernsehserien-stage2-pipeline) python/3"

print(f"Eligible programs: {len(eligible_programs_df)}")
print(f"MAX_NETWORK_CALLS: {MAX_NETWORK_CALLS}")
print(f"QUERY_DELAY_SECONDS: {QUERY_DELAY_SECONDS}")
print(f"USER_AGENT: {USER_AGENT}")

## 3) Execute Workflow
Run the pipeline once using the configured `MAX_NETWORK_CALLS` budget.
During execution, heartbeat lines are printed every minute and on each 50-network-call milestone.

In [ ]:
from process.candidate_generation.fernsehserien_de import run_pipeline_with_notebook_heartbeat

cfg = FernsehserienRunConfig(
    repo_root=ROOT,
    query_delay_seconds=QUERY_DELAY_SECONDS,
    max_network_calls=int(MAX_NETWORK_CALLS),
    max_programs=None,
    user_agent=USER_AGENT,
)

final_result = run_pipeline_with_notebook_heartbeat(config=cfg, logger=logger)
cleanup = final_result.get("cleanup", {})
print("Fragment cleanup summary:", cleanup)

fragments_path = ROOT / "data" / "20_candidate_generation" / "fernsehserien_de" / "diagnostics" / "url_fragments_observed.csv"
if fragments_path.exists():
    fragments = sorted(
        pd.read_csv(fragments_path)["fragment"].dropna().astype(str).str.strip().loc[lambda s: s.ne("")].unique().tolist()
    )
    print(f"Observed URL fragments ({len(fragments)}): {fragments}")
else:
    print("Observed URL fragments: []")

## 4) Verify Run Behavior
Validate that runtime behavior matches the configured `MAX_NETWORK_CALLS` policy.

In [ ]:
verification = {
    "eligible_programs": int(len(eligible_programs_df)),
    "programs_processed": int(final_result["programs_processed"]),
    "network_calls_used": int(final_result["network_calls_used"]),
    "max_network_calls": int(final_result["max_network_calls"]),
}

if int(MAX_NETWORK_CALLS) == 0:
    assert int(final_result["network_calls_used"]) == 0, (
        f"Cache-only run used network unexpectedly: {final_result['network_calls_used']}"
    )
elif int(MAX_NETWORK_CALLS) > 0:
    assert int(final_result["network_calls_used"]) <= int(MAX_NETWORK_CALLS), (
        "Network calls used exceeded configured budget"
    )

print("Verification:", verification)
verification

## 5) Inspect Produced Projections
Load discovered and normalized projection artifacts produced by the final pipeline pass.

In [ ]:
import json
import pandas as pd

runtime_root = ROOT / "data" / "20_candidate_generation" / "fernsehserien_de"
projection_root = runtime_root / "projections"

program_pages_df = pd.read_csv(projection_root / "program_pages.csv")
episode_index_df = pd.read_csv(projection_root / "episode_index_pages.csv")
episode_urls_df = pd.read_csv(projection_root / "episode_urls.csv")
metadata_discovered_df = pd.read_csv(projection_root / "episode_metadata_discovered.csv")
guests_discovered_df = pd.read_csv(projection_root / "episode_guests_discovered.csv")
broadcasts_discovered_df = pd.read_csv(projection_root / "episode_broadcasts_discovered.csv")
metadata_normalized_df = pd.read_csv(projection_root / "episode_metadata_normalized.csv")
guests_normalized_df = pd.read_csv(projection_root / "episode_guests_normalized.csv")
broadcasts_normalized_df = pd.read_csv(projection_root / "episode_broadcasts_normalized.csv")
summary = json.loads((projection_root / "summary.json").read_text(encoding="utf-8"))

print("Projection summary:", summary)
print("Configured max_network_calls:", int(MAX_NETWORK_CALLS))
print("Run network usage:", final_result["network_calls_used"], "/", final_result["max_network_calls"])
print("Programs processed:", final_result["programs_processed"])

In [ ]:
def _url_set(df: pd.DataFrame, preferred_col: str = "episode_url") -> set[str]:
    col = preferred_col if preferred_col in df.columns else ("url" if "url" in df.columns else None)
    if col is None:
        return set()
    return {
        v
        for v in df[col].dropna().astype(str).str.strip().tolist()
        if v
    }


def _pct(part: int, total: int) -> str:
    if total <= 0:
        return "n/a"
    return f"{(100.0 * part / total):.2f}%"


urls_all = _url_set(episode_urls_df)
urls_meta_discovered = _url_set(metadata_discovered_df)
urls_meta_normalized = _url_set(metadata_normalized_df)
urls_guest_discovered = _url_set(guests_discovered_df)
urls_guest_normalized = _url_set(guests_normalized_df)
urls_broadcast_discovered = _url_set(broadcasts_discovered_df)
urls_broadcast_normalized = _url_set(broadcasts_normalized_df)

missing_meta_discovered = sorted(urls_all - urls_meta_discovered)
missing_meta_normalized = sorted(urls_all - urls_meta_normalized)
missing_guest_discovered = sorted(urls_all - urls_guest_discovered)
missing_broadcast_discovered = sorted(urls_all - urls_broadcast_discovered)

missing_guest_with_meta = sorted(set(missing_guest_discovered) & urls_meta_discovered)
missing_guest_without_meta = sorted(set(missing_guest_discovered) - urls_meta_discovered)

print("=== Run Progress ===")
print("Programs processed:", final_result.get("programs_processed", "n/a"), "/", len(eligible_programs_df))
print(
    "Network calls used:",
    final_result.get("network_calls_used", "n/a"),
    "/",
    final_result.get("max_network_calls", "n/a"),
)
print("Projection processed_events_this_run:", summary.get("processed_events_this_run", "n/a"))
print("Fragment-tainted sequences skipped:", summary.get("fragment_tainted_sequences_skipped", 0))

print("\n=== Artifact Row Counts ===")
print("episode_urls rows:", len(episode_urls_df))
print("metadata_discovered rows:", len(metadata_discovered_df))
print("metadata_normalized rows:", len(metadata_normalized_df))
print("guests_discovered rows:", len(guests_discovered_df))
print("guests_normalized rows:", len(guests_normalized_df))
print("broadcasts_discovered rows:", len(broadcasts_discovered_df))
print("broadcasts_normalized rows:", len(broadcasts_normalized_df))

print("\n=== URL Coverage and Backlog ===")
print("Unique episode URLs total:", len(urls_all))
print(
    "Metadata discovered coverage:",
    f"{len(urls_meta_discovered)}/{len(urls_all)}",
    f"({_pct(len(urls_meta_discovered), len(urls_all))})",
    "| backlog:",
    len(missing_meta_discovered),
)
print(
    "Metadata normalized coverage:",
    f"{len(urls_meta_normalized)}/{len(urls_all)}",
    f"({_pct(len(urls_meta_normalized), len(urls_all))})",
    "| backlog:",
    len(missing_meta_normalized),
)
print(
    "Guest discovered URL coverage:",
    f"{len(urls_guest_discovered)}/{len(urls_all)}",
    f"({_pct(len(urls_guest_discovered), len(urls_all))})",
    "| no-guest URLs:",
    len(missing_guest_discovered),
)
print(
    "  no-guest URLs with metadata present:",
    len(missing_guest_with_meta),
)
print(
    "  no-guest URLs without metadata (typically unfetched/budget-blocked):",
    len(missing_guest_without_meta),
)
print(
    "Broadcast discovered URL coverage:",
    f"{len(urls_broadcast_discovered)}/{len(urls_all)}",
    f"({_pct(len(urls_broadcast_discovered), len(urls_all))})",
    "| no-broadcast URLs:",
    len(missing_broadcast_discovered),
)

program_col = "program_name" if "program_name" in episode_urls_df.columns else None
if program_col is not None:
    episode_url_col = "episode_url" if "episode_url" in episode_urls_df.columns else "url"
    episode_url_to_program = (
        episode_urls_df[[episode_url_col, program_col]]
        .copy()
        .assign(**{episode_url_col: episode_urls_df[episode_url_col].fillna("").astype(str).str.strip()})
        .drop_duplicates(subset=[episode_url_col])
        .set_index(episode_url_col)[program_col]
    )
    no_guest_programs = pd.Series(missing_guest_discovered).map(episode_url_to_program).fillna("<unknown>")
    print("\nTop programs by no-guest URL backlog:")
    print(no_guest_programs.value_counts().head(10).to_string())

print("\n=== Metadata Backlog Sample (up to 25 URLs) ===")
for url in missing_meta_discovered[:25]:
    print(url)

print("\n=== No-Guest Backlog Sample (up to 25 URLs) ===")
for url in missing_guest_discovered[:25]:
    print(url)

### Program and Index Pages

In [ ]:
program_pages_df

In [ ]:
episode_index_df

### Episode URLs

In [ ]:
episode_urls_df

### Discovered Projection Artifacts

In [ ]:
metadata_discovered_df

In [ ]:
guests_discovered_df

In [ ]:
broadcasts_discovered_df

### Normalized Projection Artifacts

In [ ]:
metadata_normalized_df

In [ ]:
guests_normalized_df

In [ ]:
broadcasts_normalized_df

In [ ]:
def _clean_str_series(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.strip()

url_col_urls = "episode_url" if "episode_url" in episode_urls_df.columns else "url"
url_col_guests = "episode_url" if "episode_url" in guests_discovered_df.columns else "url"

all_episode_urls = set(_clean_str_series(episode_urls_df[url_col_urls])) - {""}

guest_urls_all_rows = set(_clean_str_series(guests_discovered_df[url_col_guests])) - {""}

guest_name_col = "guest_name_raw" if "guest_name_raw" in guests_discovered_df.columns else None
if guest_name_col is not None:
    guest_rows_nonempty_name = guests_discovered_df[_clean_str_series(guests_discovered_df[guest_name_col]).ne("")].copy()
    guest_urls_nonempty_name = set(_clean_str_series(guest_rows_nonempty_name[url_col_guests])) - {""}
else:
    guest_rows_nonempty_name = guests_discovered_df.copy()
    guest_urls_nonempty_name = guest_urls_all_rows

no_guest_urls_all_rows = sorted(all_episode_urls - guest_urls_all_rows)
no_guest_urls_nonempty_name = sorted(all_episode_urls - guest_urls_nonempty_name)

print("=== Guest Coverage Deep Dive ===")
print("episode_urls rows:", len(episode_urls_df))
print("guests_discovered rows:", len(guests_discovered_df))
print("unique episode URLs total:", len(all_episode_urls))
print("unique guest episode URLs (all guest rows):", len(guest_urls_all_rows))
print("unique guest episode URLs (non-empty guest_name_raw):", len(guest_urls_nonempty_name))
print("no-guest URLs (all guest rows logic):", len(no_guest_urls_all_rows))
print("no-guest URLs (non-empty guest_name_raw logic):", len(no_guest_urls_nonempty_name))

if guest_name_col is not None:
    empty_name_rows = guests_discovered_df[_clean_str_series(guests_discovered_df[guest_name_col]).eq("")]
    print("guest rows with empty guest_name_raw:", len(empty_name_rows))

# Program-level concentration of no-guest URLs
program_col = "program_name" if "program_name" in episode_urls_df.columns else None
if program_col is not None:
    episode_url_to_program = (
        episode_urls_df[[url_col_urls, program_col]]
        .copy()
        .assign(**{url_col_urls: _clean_str_series(episode_urls_df[url_col_urls])})
        .drop_duplicates(subset=[url_col_urls])
        .set_index(url_col_urls)[program_col]
    )
    no_guest_programs = pd.Series(no_guest_urls_nonempty_name).map(episode_url_to_program).fillna("<unknown>")
    print("\nTop programs by no-guest URLs (non-empty guest logic):")
    print(no_guest_programs.value_counts().head(20).to_string())

print("\nSample no-guest URLs (first 30, non-empty guest logic):")
for u in no_guest_urls_nonempty_name[:30]:
    print(u)

In [ ]:
import hashlib
from process.candidate_generation.fernsehserien_de.parser import parse_episode_leaf_fields


def _cache_path_for_url_local(url: str) -> Path:
    key = hashlib.md5(url.encode("utf-8")).hexdigest()
    return runtime_root / "cache" / "pages" / f"{key}.html"

# Build sets for focused diagnostics.
url_col = "episode_url" if "episode_url" in episode_urls_df.columns else "url"
all_urls = set(episode_urls_df[url_col].dropna().astype(str).str.strip()) - {""}
meta_urls = set(metadata_discovered_df[url_col].dropna().astype(str).str.strip()) - {""}
guest_urls = set(guests_discovered_df[url_col].dropna().astype(str).str.strip()) - {""}

no_guest_urls = sorted(all_urls - guest_urls)
no_guest_with_meta = [u for u in no_guest_urls if u in meta_urls]
with_guest_with_meta = [u for u in sorted(guest_urls) if u in meta_urls]


def _inspect_url(url: str) -> dict:
    p = _cache_path_for_url_local(url)
    if not p.exists():
        return {
            "episode_url": url,
            "cache_exists": False,
            "html_has_cast_anchor": False,
            "html_has_cast_list": False,
            "parsed_guests_count": None,
        }
    html = p.read_text(encoding="utf-8", errors="replace")
    parsed = parse_episode_leaf_fields(html_text=html)
    return {
        "episode_url": url,
        "cache_exists": True,
        "html_has_cast_anchor": ("Cast-Crew" in html),
        "html_has_cast_list": ("cast-crew" in html),
        "parsed_guests_count": len(parsed.get("guests_raw", [])),
        "parser_rule": parsed.get("parser_rule", ""),
        "episode_title_raw": str(parsed.get("episode_title_raw", ""))[:80],
    }

sample_no_guest = no_guest_with_meta[:30]
sample_with_guest = with_guest_with_meta[:30]

no_guest_inspect_df = pd.DataFrame([_inspect_url(u) for u in sample_no_guest])
with_guest_inspect_df = pd.DataFrame([_inspect_url(u) for u in sample_with_guest])

print("No-guest URLs total:", len(no_guest_urls))
print("No-guest URLs with metadata:", len(no_guest_with_meta))
print("With-guest URLs with metadata:", len(with_guest_with_meta))

print("\nNo-guest sample marker rates:")
if not no_guest_inspect_df.empty:
    print("cache_exists:", int(no_guest_inspect_df["cache_exists"].sum()), "/", len(no_guest_inspect_df))
    print("html_has_cast_anchor:", int(no_guest_inspect_df["html_has_cast_anchor"].sum()), "/", len(no_guest_inspect_df))
    print("html_has_cast_list:", int(no_guest_inspect_df["html_has_cast_list"].sum()), "/", len(no_guest_inspect_df))
    print("parsed_guests_count>0:", int((no_guest_inspect_df["parsed_guests_count"].fillna(0) > 0).sum()), "/", len(no_guest_inspect_df))

print("\nWith-guest sample marker rates:")
if not with_guest_inspect_df.empty:
    print("cache_exists:", int(with_guest_inspect_df["cache_exists"].sum()), "/", len(with_guest_inspect_df))
    print("html_has_cast_anchor:", int(with_guest_inspect_df["html_has_cast_anchor"].sum()), "/", len(with_guest_inspect_df))
    print("html_has_cast_list:", int(with_guest_inspect_df["html_has_cast_list"].sum()), "/", len(with_guest_inspect_df))
    print("parsed_guests_count>0:", int((with_guest_inspect_df["parsed_guests_count"].fillna(0) > 0).sum()), "/", len(with_guest_inspect_df))

print("\nNo-guest sample detail:")
no_guest_inspect_df[["episode_url", "html_has_cast_anchor", "html_has_cast_list", "parsed_guests_count", "episode_title_raw"]].head(30)

In [ ]:
import re

re_cast_h2_id = re.compile(r'<h2[^>]*id=["\']?Cast-Crew["\']?[^>]*>', flags=re.IGNORECASE)
re_list_class = re.compile(r'class=["\'][^"\']*cast-crew[^"\']*["\']', flags=re.IGNORECASE)
re_data_event = re.compile(r'data-event-category=["\']liste-cast-crew["\']', flags=re.IGNORECASE)
re_person_href = re.compile(r'href=["\']/(personen|people)/[^"\']+["\']', flags=re.IGNORECASE)

rows = []
for url in no_guest_urls:
    p = _cache_path_for_url_local(url)
    if not p.exists():
        rows.append({"episode_url": url, "cache_exists": False})
        continue
    html = p.read_text(encoding="utf-8", errors="replace")
    rows.append(
        {
            "episode_url": url,
            "cache_exists": True,
            "has_cast_h2_id": bool(re_cast_h2_id.search(html)),
            "has_cast_list_class": bool(re_list_class.search(html)),
            "has_data_event_cast": bool(re_data_event.search(html)),
            "has_person_href": bool(re_person_href.search(html)),
        }
    )

no_guest_markup_df = pd.DataFrame(rows)

print("=== No-Guest URL Markup Audit (all no-guest URLs) ===")
print("rows:", len(no_guest_markup_df))
for c in ["cache_exists", "has_cast_h2_id", "has_cast_list_class", "has_data_event_cast", "has_person_href"]:
    if c in no_guest_markup_df.columns:
        print(f"{c}:", int(no_guest_markup_df[c].fillna(False).sum()), "/", len(no_guest_markup_df))

print("\nPattern combinations (top 10):")
combo_cols = [c for c in ["has_cast_h2_id", "has_cast_list_class", "has_data_event_cast", "has_person_href"] if c in no_guest_markup_df.columns]
if combo_cols:
    print(no_guest_markup_df[combo_cols].value_counts().head(10).to_string())

no_guest_markup_df.head(20)

In [ ]:
rows_yes = []
for url in sorted(guest_urls):
    p = _cache_path_for_url_local(url)
    if not p.exists():
        rows_yes.append({"episode_url": url, "cache_exists": False})
        continue
    html = p.read_text(encoding="utf-8", errors="replace")
    rows_yes.append(
        {
            "episode_url": url,
            "cache_exists": True,
            "has_cast_h2_id": bool(re_cast_h2_id.search(html)),
            "has_cast_list_class": bool(re_list_class.search(html)),
            "has_data_event_cast": bool(re_data_event.search(html)),
            "has_person_href": bool(re_person_href.search(html)),
        }
    )

with_guest_markup_df = pd.DataFrame(rows_yes)
print("=== With-Guest URL Markup Audit (all with-guest URLs) ===")
print("rows:", len(with_guest_markup_df))
for c in ["cache_exists", "has_cast_h2_id", "has_cast_list_class", "has_data_event_cast", "has_person_href"]:
    if c in with_guest_markup_df.columns:
        print(f"{c}:", int(with_guest_markup_df[c].fillna(False).sum()), "/", len(with_guest_markup_df))

print("\nPattern combinations (top 10):")
combo_cols = [c for c in ["has_cast_h2_id", "has_cast_list_class", "has_data_event_cast", "has_person_href"] if c in with_guest_markup_df.columns]
if combo_cols:
    print(with_guest_markup_df[combo_cols].value_counts().head(10).to_string())